In [2]:
import os
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results, ts2xy
import matplotlib.pyplot as plt
import pandas as pd
import random
from collections import deque

# =============================================================================
# PERUBAHAN UTAMA DARI VERSI LAMA:
# 1. bitrates = [0.55, 0.95, 1.67, 3.40] → 4 opsi (240p/360p/480p/720p)
# 2. action_space = Discrete(4)
# 3. Scaling trace otomatis ke max_bitrate × 1.5 = 3.40 × 1.5 = 5.10 Mbps
# 4. Normalisasi last_exec dibagi 3.0 (n-1 = 3)
# 5. SAFE_MARGIN diturunkan ke 0.3 (lompatan antar bitrate lebih kecil)
# 6. NDN_CONGESTION_CWND diturunkan ke 3.0 (threshold disesuaikan)
# 7. Total timesteps = 500.000 langkah
# 8. Reward dinormalisasi terhadap max bitrate agar tidak bias
# =============================================================================

# --- KONSTANTA GLOBAL ---
BITRATES       = [0.55, 0.95, 1.67, 3.40]   # Mbps → 240p, 360p, 480p, 720p
MAX_BITRATE    = max(BITRATES)               # 3.40 Mbps
SCALE_TARGET   = MAX_BITRATE * 1.5          # 5.10 Mbps → headroom jaringan
NUM_ACTIONS    = len(BITRATES)              # 4


class MahimahiTraceManager:
    """
    Mengelola file trace Mahimahi (.log).
    Mengonversi timestamp (ms) menjadi Throughput (Mbps).
    Scaling otomatis ke SCALE_TARGET (max_bitrate × 1.5 = 5.10 Mbps)
    agar jaringan cukup menantang untuk 4 level resolusi.
    """
    def __init__(self, folder_path="traces_folder/mahimahi"):
        self.traces = []
        PACKET_SIZE_BITS = 1500 * 8

        if os.path.exists(folder_path):
            files = [f for f in os.listdir(folder_path) if f.endswith('.log')]
            files.sort()

            for file in files:
                path = os.path.join(folder_path, file)
                try:
                    with open(path, 'r') as f:
                        timestamps_ms = [float(line.strip()) for line in f if line.strip()]

                        if timestamps_ms:
                            throughput_mbps = []
                            current_sec = 0
                            packet_count = 0

                            for ts in timestamps_ms:
                                sec = int(ts / 1000)
                                while current_sec < sec:
                                    mbps = (packet_count * PACKET_SIZE_BITS) / 1_000_000
                                    throughput_mbps.append(mbps)
                                    packet_count = 0
                                    current_sec += 1
                                packet_count += 1

                            throughput_mbps.append((packet_count * PACKET_SIZE_BITS) / 1_000_000)

                            # -------------------------------------------------------
                            # PERUBAHAN: Scaling ke SCALE_TARGET (5.10 Mbps)
                            # Sebelumnya hardcoded 12.0 Mbps.
                            # Sekarang otomatis menyesuaikan dengan max bitrate client.
                            # Dengan SCALE_TARGET = 3.40 × 1.5 = 5.10 Mbps:
                            #   - Jaringan "bagus"  → bisa memilih 720p (3.40 Mbps)
                            #   - Jaringan "sedang" → hanya cukup untuk 480p (1.67 Mbps)
                            #   - Jaringan "drop"   → turun ke 360p atau 240p
                            # -------------------------------------------------------
                            max_tp = max(throughput_mbps) if throughput_mbps else 1
                            scale_factor = SCALE_TARGET / max_tp if max_tp > 0 else 1.0

                            scaled_mbps = [max(0.1, tp * scale_factor) for tp in throughput_mbps]

                            # Pre-smoothing ringan (window=3)
                            smoothed = pd.Series(scaled_mbps).rolling(
                                window=3, min_periods=1
                            ).mean().tolist()

                            self.traces.append({"name": file, "data": smoothed})

                except Exception as e:
                    print(f"Gagal memproses file {file}: {e}")

        if not self.traces:
            print("⚠️  traces_folder tidak ditemukan. Gunakan fallback synthetic.")
            # Fallback: throughput sintetis dengan variasi
            synthetic = [3.0 + 1.5 * np.sin(i * 0.1) for i in range(500)]
            self.traces.append({"name": "synth", "data": synthetic})

        self.active_trace = None
        self.ptr = 0

        print(f"✅ {len(self.traces)} trace dimuat. Target scaling: {SCALE_TARGET:.2f} Mbps")

    def select_random_trace(self):
        self.active_trace = random.choice(self.traces)
        self.ptr = random.randint(0, max(0, len(self.active_trace["data"]) - 110))
        return self.active_trace["name"]

    def set_trace_index(self, idx):
        self.active_trace = self.traces[idx % len(self.traces)]
        self.ptr = 0
        return self.active_trace["name"]

    def get_next_bandwidth(self):
        val = self.active_trace["data"][self.ptr]
        self.ptr = (self.ptr + 1) % len(self.active_trace["data"])
        return val


class HybridStreamingEnvNDN(gym.Env):
    """
    Hybrid ABR Environment — versi 4 bitrate (240p / 360p / 480p / 720p).

    Observation Space (7 dimensi, ternormalisasi [0,1]):
      [Buffer, Mean_TP, Last_Exec, Buffer_Safety, Volatility, CWND, RTT]

    Action Space:
      Discrete(4) → indeks 0/1/2/3 → 0.55/0.95/1.67/3.40 Mbps

    Perubahan dari versi 3-bitrate:
      - NUM_ACTIONS = 4, norm_exec dibagi 3.0
      - SCALE_TARGET = 5.10 Mbps (bukan 12 Mbps)
      - SAFE_MARGIN = 0.3 Mbps (lompatan antar level lebih kecil)
      - NDN_CONGESTION_CWND = 3.0 (lebih sensitif)
      - Reward dinormalisasi: reward_bitrate = chosen / MAX_BITRATE
    """

    def __init__(self, trace_manager):
        super(HybridStreamingEnvNDN, self).__init__()
        self.trace_manager = trace_manager

        # -------------------------------------------------------
        # PERUBAHAN UTAMA: 4 level bitrate sesuai resolusi client
        # -------------------------------------------------------
        self.bitrates = BITRATES   # [0.55, 0.95, 1.67, 3.40]

        # Observation: 7 dimensi [0,1]
        self.observation_space = spaces.Box(
            low=np.zeros(7, dtype=np.float32),
            high=np.ones(7, dtype=np.float32),
            dtype=np.float32
        )

        # -------------------------------------------------------
        # PERUBAHAN: Discrete(4) — bukan Discrete(3)
        # -------------------------------------------------------
        self.action_space = spaces.Discrete(NUM_ACTIONS)

        self.state = None
        self.max_steps = 100
        self.current_step = 0
        self.tp_history = deque(maxlen=10)

        # --- Threshold Buffer ---
        self.LOW_BUFFER_THRESHOLD   = 5.0   # detik (sama)
        self.PANIC_BUFFER_THRESHOLD = 10.0   # detik (sama)

        # -------------------------------------------------------
        # PERUBAHAN: SAFE_MARGIN diturunkan dari 1.5 → 0.3
        # Karena lompatan antar bitrate sekarang lebih kecil:
        #   0.55→0.95 = +0.40 Mbps
        #   0.95→1.67 = +0.72 Mbps
        #   1.67→3.40 = +1.73 Mbps
        # Margin 1.5 Mbps terlalu besar untuk 3 step pertama.
        # -------------------------------------------------------
        self.SAFE_MARGIN = 0.3   # Mbps headroom minimum untuk upgrade

        # -------------------------------------------------------
        # PERUBAHAN: NDN_CONGESTION_CWND diturunkan dari 5.0 → 3.0
        # Karena throughput operasional lebih rendah (maks ~5 Mbps),
        # CWND kritis perlu disesuaikan agar veto tetap efektif.
        # -------------------------------------------------------
        self.NDN_CONGESTION_CWND = 3.0
        self.NDN_CACHE_HIT_RTT   = 20.0

    # ------------------------------------------------------------------
    def _get_normalized_obs(self):
        buffer, mean_tp, last_exec, _, volatility, cwnd, rtt = self.state

        norm_buffer = np.clip(buffer / 30.0, 0.0, 1.0)
        norm_tp     = np.clip(mean_tp / SCALE_TARGET, 0.0, 1.0)  # dibagi 5.10

        # -------------------------------------------------------
        # PERUBAHAN: norm_exec dibagi (NUM_ACTIONS - 1) = 3.0
        # Sebelumnya dibagi 2.0 (untuk Discrete(3))
        # -------------------------------------------------------
        norm_exec = float(last_exec) / float(NUM_ACTIONS - 1)

        safety      = max(0, buffer - self.LOW_BUFFER_THRESHOLD)
        norm_safety = np.clip(safety / 20.0, 0.0, 1.0)
        norm_vol    = np.clip(volatility, 0.0, 1.0)
        norm_cwnd   = np.clip(cwnd / 100.0, 0.0, 1.0)
        norm_rtt    = np.clip(rtt / 500.0, 0.0, 1.0)

        return np.array(
            [norm_buffer, norm_tp, norm_exec, norm_safety, norm_vol, norm_cwnd, norm_rtt],
            dtype=np.float32
        )

    # ------------------------------------------------------------------
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        if options and "trace_idx" in options:
            trace_name = self.trace_manager.set_trace_index(options["trace_idx"])
        else:
            trace_name = self.trace_manager.select_random_trace()

        initial_tp = self.trace_manager.get_next_bandwidth()

        self.tp_history.clear()
        for _ in range(self.tp_history.maxlen):
            self.tp_history.append(initial_tp)

        # State awal: buffer 15 detik, indeks 1 (360p), CWND 10, RTT 100ms
        self.state = np.array(
            [15.0, initial_tp, 1.0, 0.0, 0.0, 10.0, 100.0],
            dtype=np.float32
        )
        self.current_step = 0
        return self._get_normalized_obs(), {"trace": trace_name}

    # ------------------------------------------------------------------
    def step(self, action):
        target_idx  = int(action)
        buffer, _, current_idx, _, _, prev_cwnd, prev_rtt = self.state
        current_idx = int(current_idx)

        # 1. Update throughput & metrik jaringan
        raw_tp = self.trace_manager.get_next_bandwidth()
        self.tp_history.append(raw_tp)
        mean_tp    = np.mean(self.tp_history)
        volatility = np.std(self.tp_history) / (mean_tp + 1e-6)

        # Simulasi metrik NDN
        new_cwnd = np.clip(raw_tp * 8.0 + np.random.normal(0, 2), 2, 100)
        new_rtt  = np.clip(200.0 / (raw_tp + 0.5) + np.random.normal(0, 5), 5, 500)

        # ------------------------------------------------------------------
        # LAYER 2: NDN-AWARE DYNAMIC CONTROLLER
        # ------------------------------------------------------------------
        executed_idx = current_idx

        # Ambang buffer dinamis (makin volatil → butuh buffer lebih besar)
        dynamic_buffer_req = self.LOW_BUFFER_THRESHOLD + (volatility * 5.0)

        # A. NDN VETO — larang upgrade saat jalur macet
        is_ndn_congested = new_cwnd < self.NDN_CONGESTION_CWND
        if is_ndn_congested and target_idx > current_idx:
            target_idx = current_idx

        # B. PANIC MODE — buffer kritis, paksa turun 1 level
        if buffer < self.PANIC_BUFFER_THRESHOLD:
            executed_idx = max(0, current_idx - 1)

        # C. UPGRADE LOGIC dengan Hysteresis & NDN Health
        elif target_idx > current_idx:
            headroom = mean_tp - self.bitrates[current_idx]
            # -------------------------------------------------------
            # PERUBAHAN: SAFE_MARGIN = 0.3 Mbps (dari 1.5 Mbps)
            # Disesuaikan dengan gap antar bitrate yang lebih kecil.
            # Kondisi RTT < 300 ms tetap dipertahankan.
            # -------------------------------------------------------
            if headroom > self.SAFE_MARGIN and buffer > dynamic_buffer_req and new_rtt < 300:
                executed_idx = current_idx + 1
            else:
                executed_idx = current_idx

        # D. DOWNGRADE — langsung eksekusi tanpa syarat
        else:
            executed_idx = target_idx

        # 2. Eksekusi & hitung stalling
        executed_idx  = np.clip(executed_idx, 0, NUM_ACTIONS - 1)
        chosen_bitrate = self.bitrates[executed_idx]

        # Simulasi DASH download
        download_time = (chosen_bitrate * 5.0) / (raw_tp + 0.1)
        stalling      = max(0, download_time - buffer)
        new_buffer    = max(0, buffer - download_time) + 5.0
        new_buffer    = min(new_buffer, 30.0)
        # Contoh modifikasi untuk mengobati AI yang buta buffer
        if new_buffer < 5.0:
           reward_buffer = -20.0 # Beri penalti "sakit" karena zona kuning
        elif new_buffer < 3.0:
           reward_buffer = -50.0 # Beri penalti "kritis" karena zona merah
        else:
           reward_buffer = 5.0   # Beri bonus jika buffer sehat (> 5 detik)
        # ------------------------------------------------------------------
        # 3. REWARD FUNCTION
        # PERUBAHAN: reward bitrate dinormalisasi terhadap MAX_BITRATE
        # Sebelumnya: reward = float(chosen_bitrate)  → rentang 0.5–8.0
        # Sekarang:   reward = chosen/max             → rentang 0.0–1.0
        # Lalu diskalakan ×100 agar magnitude serupa dengan versi lama.
        # Ini mencegah bias karena selisih 0.55 vs 3.40 tidak sebesar
        # selisih 0.5 vs 8.0 pada versi lama.
        # ------------------------------------------------------------------
        norm_reward = (chosen_bitrate / MAX_BITRATE) * 100.0

        reward = norm_reward + reward_buffer
        if stalling > 0:
            reward -= (stalling * 50.0 + 20.0)   # penalti stalling (sama)
        if executed_idx != current_idx:
            reward -= 2.0                          # penalti fluktuasi (sama)

        # Update state (7 dimensi)
        self.state = np.array(
            [new_buffer, mean_tp, float(executed_idx), 0.0,
             volatility, new_cwnd, new_rtt],
            dtype=np.float32
        )

        self.current_step += 1
        done = self.current_step >= self.max_steps
        return self._get_normalized_obs(), reward, done, False, {
            "buffer":     new_buffer,
            "cwnd":       new_cwnd,
            "rtt":        new_rtt,
            "bitrate":    chosen_bitrate,
            "resolution": ["240p", "360p", "480p", "720p"][executed_idx],
            "raw_tp":     mean_tp,
            "volatility": volatility,
            "stalling":   stalling,
        }


# =============================================================================
def run_experiment():
    log_dir = "./rl_logs_4bitrate/"
    os.makedirs(log_dir, exist_ok=True)

    tm = MahimahiTraceManager(folder_path="traces_folder/mahimahi")
    if not tm.traces:
        return

    env = Monitor(HybridStreamingEnvNDN(tm), log_dir)

    print("=" * 60)
    print("  Volatility-Aware Hybrid ABR — 4 Bitrate (240p~720p)")
    print(f"  Bitrates : {BITRATES} Mbps")
    print(f"  TP Scale : {SCALE_TARGET:.2f} Mbps (max_bitrate × 1.5)")
    print(f"  Actions  : Discrete({NUM_ACTIONS})")
    print(f"  Training : 500,000 langkah")
    print("=" * 60)

    # -------------------------------------------------------
    # PERUBAHAN: total_timesteps = 500_000 (dari 300_000)
    # Action space lebih besar (4 vs 3) butuh lebih banyak
    # pengalaman agar agen menguasai semua transisi.
    # -------------------------------------------------------
    model = PPO(
        "MlpPolicy", env,
        verbose=1,
        learning_rate=0.00025,
        ent_coef=0.05,
        n_steps=2048,
        batch_size=64
    )

    model.learn(total_timesteps=500_000)
    model.save("hybrid_4bitrate_ndn_model_v3")
    print("✅ Pelatihan selesai. Model disimpan: hybrid_4bitrate_ndn_model_v3")

    # ------------------------------------------------------------------
    # EVALUASI per trace file
    # ------------------------------------------------------------------
    resolution_labels = ["240p", "360p", "480p", "720p"]
    colors_res        = ["blue", "green", "orange", "red"]

    for i in range(len(tm.traces)):
        obs, info = env.reset(options={"trace_idx": i})
        trace_name = info["trace"]
        history = []

        for _ in range(100):
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, info_step = env.step(action)
            history.append({
                "TP":         info_step["raw_tp"],
                "Volatility": info_step["volatility"],
                "Buffer":     info_step["buffer"],
                "Bitrate":    info_step["bitrate"],
                "Resolution": info_step["resolution"],
                "Stalling":   info_step["stalling"],
            })
            if done:
                break

        df = pd.DataFrame(history)

        fig, axes = plt.subplots(3, 1, figsize=(13, 12), sharex=True)

        # --- Panel 1: Throughput vs Bitrate yang dieksekusi ---
        ax1 = axes[0]
        ax1.plot(df.index, df["TP"], label="Mean TP", color="steelblue", alpha=0.4)
        ax1.step(df.index, df["Bitrate"], label="Executed Bitrate",
                 color="crimson", linewidth=2, where="post")

        # Garis referensi tiap resolusi
        for br, label, col in zip(BITRATES, resolution_labels, colors_res):
            ax1.axhline(y=br, linestyle="--", linewidth=0.8, alpha=0.5,
                        color=col, label=f"{label} ({br} Mbps)")

        ax1.set_ylabel("Mbps")
        ax1.set_title(f"4-Bitrate Hybrid ABR: {trace_name}")
        ax1.legend(fontsize=7, ncol=3)

        # --- Panel 2: Buffer Level & Stalling ---
        ax2 = axes[1]
        ax2.fill_between(df.index, df["Buffer"], color="mediumseagreen",
                         alpha=0.3, label="Buffer (detik)")
        ax2.axhline(y=5.0, color="orange", linestyle="--",
                    linewidth=1, label="Low Buffer Threshold")
        ax2.axhline(y=3.0, color="red", linestyle="--",
                    linewidth=1, label="Panic Threshold")

        # Tandai stalling
        stall_mask = df["Stalling"] > 0
        if stall_mask.any():
            ax2.scatter(df.index[stall_mask], df["Buffer"][stall_mask],
                        color="red", zorder=5, s=30, label="Stalling Event")

        ax2.set_ylabel("Buffer (detik)")
        ax2.legend(fontsize=7)

        # --- Panel 3: Volatility ---
        ax3 = axes[2]
        ax3.plot(df.index, df["Volatility"], color="purple",
                 linestyle=":", label="Volatility (CV)")
        ax3.set_ylabel("Volatility")
        ax3.set_xlabel("Segmen")
        ax3.legend(fontsize=7)

        plt.tight_layout()
        out_path = os.path.join(log_dir, f"eval_4bitrate_{trace_name}.png")
        plt.savefig(out_path, dpi=120)
        plt.close()
        print(f"  📊 Plot disimpan: {out_path}")

        # Ringkasan statistik per trace
        res_counts = df["Resolution"].value_counts()
        avg_br     = df["Bitrate"].mean()
        total_stall = df["Stalling"].sum()
        print(f"  [{trace_name}] Avg Bitrate: {avg_br:.2f} Mbps | "
              f"Total Stalling: {total_stall:.2f}s | "
              f"Distribusi: {res_counts.to_dict()}")


if __name__ == "__main__":
    run_experiment()

✅ 8 trace dimuat. Target scaling: 5.10 Mbps
  Volatility-Aware Hybrid ABR — 4 Bitrate (240p~720p)
  Bitrates : [0.55, 0.95, 1.67, 3.4] Mbps
  TP Scale : 5.10 Mbps (max_bitrate × 1.5)
  Actions  : Discrete(4)
  Training : 500,000 langkah
Using cpu device
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 100      |
|    ep_rew_mean     | 2.68e+03 |
| time/              |          |
|    fps             | 940      |
|    iterations      | 1        |
|    time_elapsed    | 2        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 100          |
|    ep_rew_mean          | 3.49e+03     |
| time/                   |              |
|    fps                  | 668          |
|    iterations           | 2            |
|    time_elapsed         | 6            |
|    total_timesteps      |

In [1]:
print("✅ Eksperimen selesai.")

✅ Eksperimen selesai.
